# Treinando algoritimo para identificar pneumopatias

## Importações

In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import seaborn as sns
import pandas as pd
from collections import Counter
import cv2
from torchvision import models
import torch.nn.functional as F
from pathlib import Path
import json

## Dispositivo que está sendo utilizado

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo utilizado: {device}")

Dispositivo utilizado: cpu


/home/jose/anaconda3/envs/anaconda-ml-ai/lib/python3.11/site-packages/torch/cuda/__init__.py:174: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


## Configuração de dispositivo


In [5]:

class COVIDQUDataset(Dataset):
    """Dataset personalizado para COVID-19 com estrutura organizada (4 classes)"""
    
    def __init__(self, dataset_root, split='train', transform=None):
        self.dataset_root = Path(dataset_root)
        self.split = split
        self.transform = transform
        self.samples = []
        
        # Definir as 4 classes conforme sua estrutura
        self.class_names = ['covid19', 'normal', 'pneumonia_bacterial', 'pneumonia_viral']
        self.class_to_idx = {class_name: idx for idx, class_name in enumerate(self.class_names)}
        
        self._load_samples()
        
        print(f"Dataset {split} carregado: {len(self.samples)} amostras")
        if len(self.samples) > 0:
            self._print_class_distribution()
        else:
            print("⚠️ AVISO: Nenhuma amostra encontrada!")
            self._debug_dataset_structure()

    def _load_samples(self):
        """Carrega amostras da estrutura organizada"""
        split_path = self.dataset_root / self.split
        
        if not split_path.exists():
            print(f"❌ ERRO: Caminho não encontrado: {split_path}")
            return
        
        print(f"Carregando dados de: {split_path}")
        
        # Para cada classe, carregar todas as imagens
        for class_name in self.class_names:
            class_path = split_path / class_name
            
            if not class_path.exists():
                print(f"⚠️ AVISO: Classe {class_name} não encontrada em {class_path}")
                continue
            
            # Buscar imagens com extensões comuns
            image_extensions = ['*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG']
            images_found = 0
            
            for ext in image_extensions:
                for img_path in class_path.glob(ext):
                    self.samples.append((str(img_path), self.class_to_idx[class_name]))
                    images_found += 1
            
            print(f"  {class_name}: {images_found} imagens carregadas")

    def _debug_dataset_structure(self):
        """Debug da estrutura do dataset"""
        print("\n=== DEBUG DA ESTRUTURA DO DATASET ===")
        print(f"Caminho base: {self.dataset_root}")
        
        if self.dataset_root.exists():
            print("Estrutura encontrada:")
            for item in self.dataset_root.iterdir():
                if item.is_dir():
                    print(f"  📁 {item.name}/")
                    for subitem in item.iterdir():
                        if subitem.is_dir():
                            # Contar imagens no subdiretório
                            img_count = sum(1 for f in subitem.glob('*') if f.suffix.lower() in ['.png', '.jpg', '.jpeg'])
                            print(f"    📁 {subitem.name}/ ({img_count} imagens)")
                        else:
                            print(f"    📄 {subitem.name}")
        else:
            print(f"❌ Caminho não existe: {self.dataset_root}")

    def _print_class_distribution(self):
        """Imprime distribuição das classes"""
        if not self.samples:
            print("Nenhuma amostra para mostrar distribuição")
            return
            
        labels = [sample[1] for sample in self.samples]
        class_counts = Counter(labels)
        print(f"\nDistribuição das classes ({self.split}):")
        total = len(self.samples)
        
        for class_idx, class_name in enumerate(self.class_names):
            count = class_counts.get(class_idx, 0)
            percentage = (count / total) * 100 if total > 0 else 0
            print(f"  {class_name}: {count} amostras ({percentage:.1f}%)")

    def get_class_weights(self):
        """Calcula pesos das classes para balanceamento"""
        if not self.samples:
            return torch.ones(len(self.class_names))
            
        labels = [sample[1] for sample in self.samples]
        class_counts = Counter(labels)
        total_samples = len(self.samples)
        
        weights = []
        for i in range(len(self.class_names)):
            if i in class_counts and class_counts[i] > 0:
                weight = total_samples / (len(self.class_names) * class_counts[i])
                weights.append(weight)
            else:
                weights.append(1.0)  # Peso padrão para classes ausentes
        
        return torch.FloatTensor(weights)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if idx >= len(self.samples):
            raise IndexError(f"Index {idx} out of range for dataset with {len(self.samples)} samples")
            
        img_path, label = self.samples[idx]
        
        try:
            image = Image.open(img_path).convert("RGB")
            
            if self.transform:
                image = self.transform(image)
            else:
                # Transformação básica se nenhuma for especificada
                image = transforms.ToTensor()(image)
                image = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(image)

            return image, label
            
        except Exception as e:
            print(f"Erro ao carregar imagem {img_path}: {e}")
            # Retornar imagem preta em caso de erro
            dummy_image = torch.zeros(3, 224, 224)
            return dummy_image, label
        
def get_transforms(phase='train'):
    """Define transformações específicas para cada fase"""
    
    if phase == 'train':
        return transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=5),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                 std=[0.229, 0.224, 0.225])
        ])
    else:
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                 std=[0.229, 0.224, 0.225])
        ])

In [6]:
class COVIDClassificationCNN(nn.Module):
    """CNN melhorada para classificação COVID-19 (4 classes)"""
    
    def __init__(self, num_classes=4, pretrained=True, dropout_rate=0.3):
        super(COVIDClassificationCNN, self).__init__()
        
        # Backbone pré-treinado
        self.backbone = models.resnet50(weights='IMAGENET1K_V1' if pretrained else None)
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        
        # Classificador ajustado para 4 classes
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )
        
        self._initialize_weights()

    def _initialize_weights(self):
        """Inicialização dos pesos"""
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

def create_balanced_dataloader(dataset, batch_size, is_train=True):
    """Cria DataLoader com balanceamento de classes"""
    if not dataset.samples:
        return None
        
    if is_train and len(dataset) > 0:
        # Calcular pesos para cada amostra
        labels = [sample[1] for sample in dataset.samples]
        class_counts = Counter(labels)
        
        # Peso inversamente proporcional à frequência da classe
        weights = []
        for label in labels:
            weights.append(1.0 / class_counts[label])
        
        # Criar sampler balanceado
        sampler = WeightedRandomSampler(
            weights=weights,
            num_samples=len(weights),
            replacement=True
        )
        
        return DataLoader(dataset, batch_size=batch_size, sampler=sampler, 
                         num_workers=2, pin_memory=True)
    else:
        return DataLoader(dataset, batch_size=batch_size, shuffle=False,
                         num_workers=2, pin_memory=True)

In [7]:
class ModelTrainer:
    """Trainer melhorado com métricas por classe"""
    
    def __init__(self, model, train_loader, val_loader, criterion, optimizer, 
                 scheduler=None, device='cpu', class_names=None):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        self.class_names = class_names or ['Class 0', 'Class 1', 'Class 2', 'Class 3']
        
        # Histórico
        self.train_losses = []
        self.train_accuracies = []
        self.val_losses = []
        self.val_accuracies = []
        self.per_class_metrics = []
        
    def train_epoch(self):
        """Treina uma época com métricas detalhadas"""
        self.model.train()
        running_loss = 0.0
        all_predictions = []
        all_labels = []
        
        for batch_idx, (images, labels) in enumerate(self.train_loader):
            images, labels = images.to(self.device), labels.to(self.device)
            
            outputs = self.model(images)
            loss = self.criterion(outputs, labels)
            
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            if (batch_idx + 1) % 100 == 0:
                print(f'  Batch {batch_idx+1}/{len(self.train_loader)}, Loss: {loss.item():.4f}')
        
        epoch_loss = running_loss / len(self.train_loader)
        epoch_acc = np.mean(np.array(all_predictions) == np.array(all_labels))
        
        # Calcular métricas por classe
        class_acc = self._calculate_per_class_accuracy(all_labels, all_predictions)
        
        return epoch_loss, epoch_acc, class_acc
    
    def validate_epoch(self):
        """Valida uma época"""
        self.model.eval()
        running_loss = 0.0
        all_predictions = []
        all_labels = []
        
        with torch.no_grad():
            for images, labels in self.val_loader:
                images, labels = images.to(self.device), labels.to(self.device)
                
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                
                running_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                all_predictions.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        epoch_loss = running_loss / len(self.val_loader)
        epoch_acc = np.mean(np.array(all_predictions) == np.array(all_labels))
        
        class_acc = self._calculate_per_class_accuracy(all_labels, all_predictions)
        
        return epoch_loss, epoch_acc, class_acc
    
    def _calculate_per_class_accuracy(self, labels, predictions):
        """Calcula acurácia por classe"""
        class_acc = {}
        for i, class_name in enumerate(self.class_names):
            class_mask = np.array(labels) == i
            if np.sum(class_mask) > 0:
                class_predictions = np.array(predictions)[class_mask]
                class_labels = np.array(labels)[class_mask]
                class_acc[class_name] = np.mean(class_predictions == class_labels)
            else:
                class_acc[class_name] = 0.0
        return class_acc
    
    def train(self, num_epochs, early_stopping_patience=10):
        """Treinamento com early stopping e métricas por classe"""
        best_val_acc = 0.0
        patience_counter = 0
        
        print(f"Iniciando treinamento por {num_epochs} épocas...")
        print("-" * 80)
        
        for epoch in range(num_epochs):
            print(f'Época {epoch+1}/{num_epochs}')
            
            # Treinamento
            train_loss, train_acc, train_class_acc = self.train_epoch()
            
            # Validação
            if self.val_loader is not None:
                val_loss, val_acc, val_class_acc = self.validate_epoch()
            else:
                val_loss, val_acc, val_class_acc = train_loss, train_acc, train_class_acc
                print("⚠️ Sem dados de validação, usando métricas de treino")
            
            # Scheduler
            if self.scheduler:
                old_lr = self.optimizer.param_groups[0]['lr']
                self.scheduler.step(val_loss)
                new_lr = self.optimizer.param_groups[0]['lr']
                if new_lr != old_lr:
                    print(f'Learning rate: {old_lr:.6f} -> {new_lr:.6f}')
            
            # Salvar histórico
            self.train_losses.append(train_loss)
            self.train_accuracies.append(train_acc)
            self.val_losses.append(val_loss)
            self.val_accuracies.append(val_acc)
            self.per_class_metrics.append({
                'epoch': epoch,
                'train_class_acc': train_class_acc,
                'val_class_acc': val_class_acc
            })
            
            print(f'Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}')
            print(f'Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}')
            
            # Mostrar acurácia por classe
            print("Acurácia por classe (Validação):")
            for class_name, acc in val_class_acc.items():
                print(f"  {class_name}: {acc:.4f}")
            
            # Early stopping
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                patience_counter = 0
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'val_acc': val_acc,
                    'val_loss': val_loss,
                    'class_acc': val_class_acc
                }, 'best_covid_model.pth')
                print(f'✓ Melhor modelo salvo! Val Acc: {val_acc:.4f}')
            else:
                patience_counter += 1
            
            if patience_counter >= early_stopping_patience:
                print(f'Early stopping após {early_stopping_patience} épocas sem melhoria')
                break
            
            print("-" * 80)
        
        return best_val_acc

    def plot_training_history(self):
        """Plot do histórico de treinamento"""
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
        
        epochs = range(1, len(self.train_losses) + 1)
        
        # Loss
        ax1.plot(epochs, self.train_losses, 'b-', label='Train Loss', linewidth=2)
        ax1.plot(epochs, self.val_losses, 'r-', label='Val Loss', linewidth=2)
        ax1.set_title('Model Loss', fontsize=14, fontweight='bold')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Accuracy
        ax2.plot(epochs, self.train_accuracies, 'b-', label='Train Acc', linewidth=2)
        ax2.plot(epochs, self.val_accuracies, 'r-', label='Val Acc', linewidth=2)
        ax2.set_title('Model Accuracy', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Accuracy')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # Acurácia por classe ao longo do tempo (validação)
        if self.per_class_metrics:
            for class_name in self.class_names:
                class_accs = [metric['val_class_acc'].get(class_name, 0) for metric in self.per_class_metrics]
                ax3.plot(epochs, class_accs, label=f'{class_name}', linewidth=2, marker='o', markersize=4)
            
            ax3.set_title('Validation Accuracy by Class', fontsize=14, fontweight='bold')
            ax3.set_xlabel('Epoch')
            ax3.set_ylabel('Class Accuracy')
            ax3.legend()
            ax3.grid(True, alpha=0.3)
        
        # Learning curve comparison
        ax4.plot(epochs, np.array(self.train_accuracies) - np.array(self.val_accuracies), 
                'g-', label='Train-Val Gap', linewidth=2)
        ax4.axhline(y=0, color='k', linestyle='--', alpha=0.5)
        ax4.set_title('Overfitting Detection', fontsize=14, fontweight='bold')
        ax4.set_xlabel('Epoch')
        ax4.set_ylabel('Accuracy Gap')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
        plt.show()

In [8]:
def evaluate_model(model, test_loader, class_names, device):
    """Avaliação completa do modelo"""
    model.eval()
    all_predictions = []
    all_labels = []
    all_probs = []
    
    print("Avaliando modelo no conjunto de teste...")
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            probs = F.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    # Métricas básicas
    overall_accuracy = np.mean(np.array(all_predictions) == np.array(all_labels))
    print(f"Acurácia geral: {overall_accuracy:.4f}")
    
    # Relatório de classificação
    print("\nRelatório de Classificação:")
    print("=" * 60)
    report = classification_report(all_labels, all_predictions, 
                                 target_names=class_names, 
                                 digits=4)
    print(report)
    
    # Matriz de confusão
    cm = confusion_matrix(all_labels, all_predictions)
    
    plt.figure(figsize=(15, 6))
    
    # Matriz de confusão - valores absolutos
    plt.subplot(1, 2, 1)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix (Absolute)', fontsize=14, fontweight='bold')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    
    # Matriz de confusão - normalizada
    plt.subplot(1, 2, 2)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm_normalized, annot=True, fmt='.3f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # AUC-ROC para classificação multiclasse (4 classes)
    try:
        # Converter para one-hot encoding
        y_true_onehot = np.zeros((len(all_labels), len(class_names)))
        for i, label in enumerate(all_labels):
            y_true_onehot[i, label] = 1
        
        # Calcular AUC por classe
        auc_scores = {}
        for i, class_name in enumerate(class_names):
            if np.sum(y_true_onehot[:, i]) > 0:  # Verificar se a classe existe
                auc = roc_auc_score(y_true_onehot[:, i], np.array(all_probs)[:, i])
                auc_scores[class_name] = auc
                print(f"AUC-ROC {class_name}: {auc:.4f}")
        
        # AUC médio
        if auc_scores:
            mean_auc = np.mean(list(auc_scores.values()))
            print(f"AUC-ROC médio: {mean_auc:.4f}")
            
    except Exception as e:
        print(f"Erro ao calcular AUC-ROC: {e}")
    
    return {
        'accuracy': overall_accuracy,
        'predictions': all_predictions,
        'labels': all_labels,
        'probabilities': all_probs,
        'confusion_matrix': cm,
        'classification_report': report
    }

In [9]:
def visualize_sample_predictions(model, test_dataset, device, class_names, num_samples=16):
    """Visualiza predições em amostras do conjunto de teste"""
    model.eval()
    
    # Selecionar amostras aleatórias
    indices = np.random.choice(len(test_dataset), min(num_samples, len(test_dataset)), replace=False)
    
    fig, axes = plt.subplots(4, 4, figsize=(16, 16))
    axes = axes.ravel()
    
    with torch.no_grad():
        for i, idx in enumerate(indices):
            if i >= num_samples:
                break
                
            image, true_label = test_dataset[idx]
            image_batch = image.unsqueeze(0).to(device)
            
            output = model(image_batch)
            prob = F.softmax(output, dim=1)
            _, predicted = torch.max(output, 1)
            
            # Desnormalizar imagem para visualização
            mean = torch.tensor([0.485, 0.456, 0.406])
            std = torch.tensor([0.229, 0.224, 0.225])
            image_denorm = image.clone()
            for t, m, s in zip(image_denorm, mean, std):
                t.mul_(s).add_(m)
            image_denorm = torch.clamp(image_denorm, 0, 1)
            
            # Plot
            axes[i].imshow(image_denorm.permute(1, 2, 0))
            axes[i].axis('off')
            
            # Título com predição
            pred_class = class_names[predicted.item()]
            true_class = class_names[true_label]
            confidence = prob[0, predicted.item()].item()
            
            color = 'green' if predicted.item() == true_label else 'red'
            title = f'True: {true_class}\nPred: {pred_class}\nConf: {confidence:.3f}'
            axes[i].set_title(title, fontsize=10, color=color, fontweight='bold')
    
    # Remover eixos vazios
    for i in range(len(indices), len(axes)):
        axes[i].axis('off')
    
    plt.suptitle('Sample Predictions', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('sample_predictions.png', dpi=300, bbox_inches='tight')
    plt.show()

In [10]:
def save_model_info(model, results, class_names, save_path="model_info.json"):
    """Salva informações do modelo e resultados"""
    model_info = {
        'model_architecture': str(model),
        'num_parameters': sum(p.numel() for p in model.parameters()),
        'trainable_parameters': sum(p.numel() for p in model.parameters() if p.requires_grad),
        'class_names': class_names,
        'num_classes': len(class_names),
        'test_accuracy': float(results['accuracy']),
        'confusion_matrix': results['confusion_matrix'].tolist(),
        'classification_report': results['classification_report']
    }
    
    with open(save_path, 'w') as f:
        json.dump(model_info, f, indent=2)
    
    print(f"Informações do modelo salvas em: {save_path}")

In [ ]:
def main():
    """Função principal"""
    print("=" * 80)
    print("SISTEMA DE CLASSIFICAÇÃO COVID-19 COM RAIOS-X (4 CLASSES)")
    print("=" * 80)
    
    # Caminho para seu dataset organizado
    dataset_root = "dataset_organizado"  # Ajuste este caminho conforme necessário
    
    if not os.path.exists(dataset_root):
        print(f"❌ ERRO: Dataset não encontrado em {dataset_root}")
        print("Por favor, certifique-se de que o dataset está no caminho correto.")
        return
    
    try:
        # 1. Definir transformações
        train_transform = get_transforms('train')
        val_transform = get_transforms('val')
        test_transform = get_transforms('test')
        
        # 2. Criar datasets
        print("\nCriando datasets...")
        train_dataset = COVIDQUDataset(dataset_root, 'train', train_transform)
        val_dataset = COVIDQUDataset(dataset_root, 'val', val_transform)
        test_dataset = COVIDQUDataset(dataset_root, 'test', test_transform)
        
        # Verificar se os datasets foram carregados corretamente
        if len(train_dataset) == 0:
            print("❌ ERRO: Dataset de treino vazio! Verifique a estrutura do dataset.")
            return
        
        print(f"✓ Datasets criados com sucesso!")
        print(f"  Treino: {len(train_dataset)} amostras")
        print(f"  Validação: {len(val_dataset)} amostras")  
        print(f"  Teste: {len(test_dataset)} amostras")
        
        # 3. Criar DataLoaders
        batch_size = 32  # Aumentado para aproveitar o dataset maior
        train_loader = create_balanced_dataloader(train_dataset, batch_size, is_train=True)
        val_loader = create_balanced_dataloader(val_dataset, batch_size, is_train=False) if len(val_dataset) > 0 else None
        test_loader = create_balanced_dataloader(test_dataset, batch_size, is_train=False) if len(test_dataset) > 0 else None
        
        if train_loader is None:
            print("❌ ERRO: Não foi possível criar o DataLoader de treino!")
            return
            
        # 4. Criar modelo para 4 classes
        print("\nCriando modelo...")
        model = COVIDClassificationCNN(num_classes=4, pretrained=True, dropout_rate=0.3)
        model = model.to(device)
        
        # Calcular pesos das classes para loss balanceado
        class_weights = train_dataset.get_class_weights().to(device)
        print(f"Pesos das classes: {class_weights}")
        
        # 5. Definir loss, optimizer e scheduler
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=5 #verbose=True
        )
        
        # 6. Criar trainer
        trainer = ModelTrainer(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
            class_names=train_dataset.class_names
        )
        
        # 7. Treinar modelo
        print("\nIniciando treinamento...")
        num_epochs = 50
        early_stopping_patience = 10
        
        best_val_acc = trainer.train(
            num_epochs=num_epochs,
            early_stopping_patience=early_stopping_patience
        )
        
        print(f"\n✓ Treinamento concluído! Melhor acurácia de validação: {best_val_acc:.4f}")
        
        # 8. Plotar histórico de treinamento
        print("\nGerando gráficos do treinamento...")
        trainer.plot_training_history()
        
        # 9. Carregar melhor modelo para avaliação
        if os.path.exists('best_covid_model.pth'):
            print("\nCarregando melhor modelo para avaliação...")
            checkpoint = torch.load('best_covid_model.pth', map_location=device)
            model.load_state_dict(checkpoint['model_state_dict'])
            print(f"✓ Modelo carregado! Val Acc: {checkpoint['val_acc']:.4f}")
        else:
            print("⚠️ Arquivo do melhor modelo não encontrado, usando modelo atual")
        
        # 10. Avaliação no conjunto de teste
        if test_loader is not None and len(test_dataset) > 0:
            print("\nAvaliando modelo no conjunto de teste...")
            results = evaluate_model(model, test_loader, train_dataset.class_names, device)
            
            # 11. Visualizar predições de amostra
            print("\nVisualizando predições de amostra...")
            visualize_sample_predictions(
                model, test_dataset, device, 
                train_dataset.class_names, num_samples=16
            )
            
            # 12. Salvar informações do modelo
            save_model_info(model, results, train_dataset.class_names)
            
        else:
            print("⚠️ Conjunto de teste não disponível para avaliação")
            # Avaliar no conjunto de validação como alternativa
            if val_loader is not None:
                print("Avaliando no conjunto de validação...")
                results = evaluate_model(model, val_loader, train_dataset.class_names, device)
                visualize_sample_predictions(
                    model, val_dataset, device, 
                    train_dataset.class_names, num_samples=16
                )
                save_model_info(model, results, train_dataset.class_names)
        
        # 13. Análise adicional dos resultados
        print("\n" + "="*80)
        print("RESUMO FINAL DOS RESULTADOS")
        print("="*80)
        
        # Informações do modelo
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        
        print(f"Arquitetura: ResNet50 + Classificador personalizado")
        print(f"Total de parâmetros: {total_params:,}")
        print(f"Parâmetros treináveis: {trainable_params:,}")
        print(f"Classes: {', '.join(train_dataset.class_names)}")
        
        # Estatísticas do dataset
        print(f"\nEstatísticas do Dataset:")
        print(f"  Treino: {len(train_dataset)} amostras")
        print(f"  Validação: {len(val_dataset)} amostras")
        print(f"  Teste: {len(test_dataset)} amostras")
        
        # Melhor acurácia alcançada
        if 'results' in locals():
            print(f"\nMelhor acurácia no teste: {results['accuracy']:.4f}")
        else:
            print(f"\nMelhor acurácia de validação: {best_val_acc:.4f}")
        
        print("\n✓ Pipeline completo executado com sucesso!")
        print("Arquivos gerados:")
        print("  - best_covid_model.pth (melhor modelo)")
        print("  - training_history.png (histórico de treinamento)")
        print("  - confusion_matrix.png (matriz de confusão)")
        print("  - sample_predictions.png (predições de amostra)")
        print("  - model_info.json (informações do modelo)")
        
    except Exception as e:
        print(f"\n❌ ERRO durante a execução: {e}")
        import traceback
        traceback.print_exc()
        return
    
    print("\n" + "="*80)
    print("EXECUÇÃO CONCLUÍDA!")
    print("="*80)

def load_and_predict(model_path, image_path, class_names, device):
    """Função para carregar modelo treinado e fazer predição em nova imagem"""
    
    # Carregar modelo
    model = COVIDClassificationCNN(num_classes=len(class_names))
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    
    # Preprocessar imagem
    transform = get_transforms('test')
    
    try:
        image = Image.open(image_path).convert("RGB")
        image_tensor = transform(image).unsqueeze(0).to(device)
        
        with torch.no_grad():
            output = model(image_tensor)
            probabilities = F.softmax(output, dim=1)
            _, predicted = torch.max(output, 1)
        
        # Resultados
        pred_class = class_names[predicted.item()]
        confidence = probabilities[0, predicted.item()].item()
        
        print(f"Predição: {pred_class}")
        print(f"Confiança: {confidence:.4f}")
        print("\nProbabilidades por classe:")
        for i, class_name in enumerate(class_names):
            prob = probabilities[0, i].item()
            print(f"  {class_name}: {prob:.4f}")
        
        return {
            'predicted_class': pred_class,
            'confidence': confidence,
            'probabilities': {class_names[i]: probabilities[0, i].item() 
                            for i in range(len(class_names))}
        }
        
    except Exception as e:
        print(f"Erro ao processar imagem: {e}")
        return None

def create_inference_example():
    """Exemplo de como usar o modelo treinado para inferência"""
    
    print("\n" + "="*60)
    print("EXEMPLO DE INFERÊNCIA COM MODELO TREINADO")
    print("="*60)
    
    class_names = ['covid19', 'normal', 'pneumonia_bacterial', 'pneumonia_viral']
    model_path = 'best_covid_model.pth'
    
    if not os.path.exists(model_path):
        print(f"❌ Modelo não encontrado em {model_path}")
        print("Execute o treinamento primeiro!")
        return
    
    print("Para usar o modelo treinado:")
    print("1. Tenha uma imagem de raio-X do tórax")
    print("2. Use a função load_and_predict:")
    print()
    print("Exemplo de uso:")
    print("```python")
    print("# Fazer predição em nova imagem")
    print("image_path = 'caminho/para/sua/imagem.jpg'")
    print("device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')")
    print("class_names = ['covid19', 'normal', 'pneumonia_bacterial', 'pneumonia_viral']")
    print()
    print("result = load_and_predict(")
    print("    model_path='best_covid_model.pth',")
    print("    image_path=image_path,")
    print("    class_names=class_names,")
    print("    device=device")
    print(")")
    print()
    print("if result:")
    print("    print(f'Classe predita: {result[\"predicted_class\"]}')") 
    print("    print(f'Confiança: {result[\"confidence\"]:.4f}')")
    print("```")

if __name__ == "__main__":
    # Executar pipeline principal
    main()
    
    # Mostrar exemplo de inferência
    create_inference_example()

SISTEMA DE CLASSIFICAÇÃO COVID-19 COM RAIOS-X (4 CLASSES)

Criando datasets...
Carregando dados de: dataset_organizado/train
  covid19: 11497 imagens carregadas
  normal: 11536 imagens carregadas
  pneumonia_bacterial: 11518 imagens carregadas
  pneumonia_viral: 11474 imagens carregadas
Dataset train carregado: 46025 amostras

Distribuição das classes (train):
  covid19: 11497 amostras (25.0%)
  normal: 11536 amostras (25.1%)
  pneumonia_bacterial: 11518 amostras (25.0%)
  pneumonia_viral: 11474 amostras (24.9%)
Carregando dados de: dataset_organizado/val
  covid19: 2356 imagens carregadas
  normal: 2364 imagens carregadas
  pneumonia_bacterial: 2361 imagens carregadas
  pneumonia_viral: 2351 imagens carregadas
Dataset val carregado: 9432 amostras

Distribuição das classes (val):
  covid19: 2356 amostras (25.0%)
  normal: 2364 amostras (25.1%)
  pneumonia_bacterial: 2361 amostras (25.0%)
  pneumonia_viral: 2351 amostras (24.9%)
Carregando dados de: dataset_organizado/test
  covid19: 23

/home/jose/anaconda3/envs/anaconda-ml-ai/lib/python3.11/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
